In [1]:
!pip install -qU langchain langchain-groq pydantic langsmith

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.2/108.2 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 471.9/471.9 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 59.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 373.3/373.3 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 8.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.50.0 requires pydantic<=2.12.3,>=2.0, but you have pydantic 2.13.1 which is incompatible.


In [2]:
!pip install -qU pypdf langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 335.6/335.6 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 66.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [3]:
import os
from google.colab import userdata

# 1. Load API Keys
# Ensure your secrets are named exactly like this in the Colab Secrets tab
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
os.environ["LANGCHAIN_API_KEY"] = userdata.get('LANGCHAIN_API_KEY')

# 2. Mandatory LangSmith Tracing Setup
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "Resume_Screening_Assignment"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"

print("✅ Environment Variables and API Keys Configured Successfully!")

✅ Environment Variables and API Keys Configured Successfully!


In [9]:
from langchain_core.prompts import ChatPromptTemplate

system_prompt = """You are an expert technical recruiter evaluating candidates for a job.
You must strictly follow these rules:
1. Do NOT assume skills or experience not explicitly present in the resume.
2. Avoid any hallucinated assumptions.
3. Be objective. A strong match should score 80-100, average 40-79, weak 0-39.
4. The 'score' output MUST be a raw integer number (e.g., 90). Do NOT wrap the score in quotes.

Job Description:
{job_description}
"""

human_prompt = """Please evaluate the following resume:
Resume:
{resume_text}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", human_prompt)
])

# Re-build the pipeline with the updated prompt
screening_pipeline = prompt | structured_llm

In [10]:
import pandas as pd
from google.colab import files
from langchain_community.document_loaders import PyPDFLoader
from IPython.display import display

print("=== Final Batch Evaluation ===")
print("👇 Click 'Choose Files' and select ALL THREE PDF resumes at once:")
uploaded_files = files.upload()

evaluation_results = []

if uploaded_files:
    print(f"\n✅ {len(uploaded_files)} files uploaded! Starting the AI screening process...\n")
    print("=" * 60)

    for filename in uploaded_files.keys():
        print(f"🔄 Currently Evaluating: {filename}\n")

        try:
            # 1. Extract text
            loader = PyPDFLoader(filename)
            pages = loader.load()
            extracted_resume_text = "\n".join([page.page_content for page in pages])

            # 2. Run the LangChain Pipeline
            result = screening_pipeline.invoke(
                {"job_description": JOB_DESCRIPTION, "resume_text": extracted_resume_text},
                config={"tags": ["final_batch_run"]}
            )

            # 3. Print individual results
            print(f"🎯 Score: {result.score}/100")
            print(f"🛠️ Key Skills Found: {', '.join(result.extracted_skills)}")
            print(f"🧠 Match Analysis: {result.matching_analysis}")
            print(f"📝 Final Explanation: {result.explanation}\n")
            print("-" * 60)

            # 4. Store for the leaderboard
            evaluation_results.append({
                "Candidate File": filename,
                "Fit Score": result.score,
                "Key Skills": ", ".join(result.extracted_skills[:4]) + "...",
                "Category": "Strong Match" if result.score >= 80 else ("Average Match" if result.score >= 40 else "Weak Match")
            })

        except Exception as e:
            print(f"❌ Error processing {filename}: {str(e)}\n")

    print("\n=== 🎉 All Evaluations Complete! ===")

    # ==========================================
    # Display Leaderboard and Final Result
    # ==========================================
    if evaluation_results:
        df = pd.DataFrame(evaluation_results)
        # Sort by highest score
        df = df.sort_values(by="Fit Score", ascending=False).reset_index(drop=True)

        print("\n📊 Final Candidate Leaderboard:")
        display(df)

        # Determine the winner (the one at the top of the sorted list)
        top_candidate = df.iloc[0]

        print("\n" + "=" * 60)
        print(f"🏆 FINAL HIRING DECISION 🏆")
        print("=" * 60)
        print(f"The candidate most likely to get the role is: {top_candidate['Candidate File']}")
        print(f"With an outstanding Fit Score of: {top_candidate['Fit Score']}/100")
        print("This candidate possesses the necessary GenAI pipeline experience, robust toolset, and development background required for the position.")

else:
    print("No files were uploaded. Please run the cell again to try.")

=== Final Batch Evaluation ===
👇 Click 'Choose Files' and select ALL THREE PDF resumes at once:


Saving weak candidate resume.pdf to weak candidate resume (3).pdf
Saving avg candidate resume.pdf to avg candidate resume (3).pdf
Saving Strong candidate resume.pdf to Strong candidate resume (3).pdf

✅ 3 files uploaded! Starting the AI screening process...

🔄 Currently Evaluating: weak candidate resume (3).pdf

🎯 Score: 0/100
🛠️ Key Skills Found: Microsoft Excel, Microsoft Word, Google Workspace, Data Entry, Time Management, Scheduling, Customer Communication, Inventory Tracking
🧠 Match Analysis: The candidate's experience and skills do not match the job requirements for a Generative AI Engineer. The candidate lacks experience in software development, AI projects, and relevant technical skills such as Python, LangChain, Large Language Models (LLMs), FastAPI, and Vector Databases.
📝 Final Explanation: The candidate's background is in administrative and data operations, with no evidence of experience in software development or AI-related fields. The skills listed are not relevant to the

,Candidate File,Fit Score,Key Skills,Category
0,Strong candidate resume (3).pdf,92,"Python, LangChain, Large Language Models (LLMs...",Strong Match
1,avg candidate resume (3).pdf,30,"Python, JavaScript, HTML/CSS, Django...",Weak Match
2,weak candidate resume (3).pdf,0,"Microsoft Excel, Microsoft Word, Google Worksp...",Weak Match



🏆 FINAL HIRING DECISION 🏆
The candidate most likely to get the role is: Strong candidate resume (3).pdf
With an outstanding Fit Score of: 92/100
This candidate possesses the necessary GenAI pipeline experience, robust toolset, and development background required for the position.
